# 🎨 ERNIE-Image-Turbo — Slide Generator

**Setup:** Enable **GPU T4 x2** in Session Options (right panel) before running.

- Model: `baidu/ERNIE-Image-Turbo` (8 steps, fast)
- Resolution: 1024×1024
- Speed: ~15s per image on T4 x2


## 1. Install Dependencies

In [1]:
!pip install -q git+https://github.com/huggingface/diffusers
!pip install -q transformers accelerate sentencepiece Pillow
print('✅ Done')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Done


## 2. Load Model

In [2]:
import torch
from diffusers import ErnieImagePipeline
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import os, glob, zipfile, textwrap

os.makedirs('/kaggle/working/slides_output', exist_ok=True)

# Check GPUs available
print(f'GPUs available: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

print('\nLoading ERNIE-Image-Turbo across both GPUs...')
pipe = ErnieImagePipeline.from_pretrained(
    'baidu/ERNIE-Image-Turbo',
    torch_dtype=torch.bfloat16,
    device_map='balanced',   # ← splits across both GPUs automatically
)

print('✅ Model loaded!')
print(f'   GPU 0 used: {torch.cuda.memory_allocated(0) / 1e9:.1f} GB')
print(f'   GPU 1 used: {torch.cuda.memory_allocated(1) / 1e9:.1f} GB')

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


GPUs available: 2
  GPU 0: 15.6 GB
  GPU 1: 15.6 GB

Loading ERNIE-Image-Turbo across both GPUs...


Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
The config attributes {'lora_rank': 4, 'use_lora': False} were passed to ErnieImageTransformer2DModel, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}


Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/237 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The config attributes {'lora_rank': 4, 'use_lora': False} were passed to ErnieImageTransformer2DModel, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Model loaded!
   GPU 0 used: 7.7 GB
   GPU 1 used: 7.8 GB


## 3. Helper Functions

In [3]:
def generate_image(prompt, seed=42, width=512, height=512):
    generator = torch.Generator('cpu').manual_seed(seed)  # ← was 'cuda'
    image = pipe(
        prompt=prompt,
        height=height,
        width=width,
        num_inference_steps=8,
        guidance_scale=1.0,
        use_pe=True,
        generator=generator,
    ).images[0]
    return image


def add_text_overlay(image, title, bullets):
    """Add title and bullet points on top of image."""
    W, H = image.size

    # Dark gradient overlay
    overlay = Image.new('RGBA', (W, H), (0, 0, 0, 0))
    draw_ov = ImageDraw.Draw(overlay)
    draw_ov.rectangle([(0, 0),   (W, H)],        fill=(0, 0, 0, 110))
    draw_ov.rectangle([(0, H*0.2), (W, H)],      fill=(0, 0, 0, 150))
    image = Image.alpha_composite(image.convert('RGBA'), overlay).convert('RGB')
    draw = ImageDraw.Draw(image)

    # Fonts
    try:
        font_title  = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 56)
        font_bullet = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 36)
        font_dot    = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 36)
    except:
        font_title = font_bullet = font_dot = ImageFont.load_default()

    # Title
    draw.text((60, 40), title, fill=(255, 255, 255), font=font_title)
    title_w = draw.textlength(title, font=font_title)
    draw.rectangle([(60, 108), (60 + title_w, 113)], fill='#f9a825')

    # Bullets
    y = 160
    for b in bullets:
        draw.text((60, y), '▶', fill='#4fc3f7', font=font_dot)
        for line in textwrap.wrap(b, width=55):
            draw.text((115, y), line, fill=(230, 230, 230), font=font_bullet)
            y += 48
        y += 16

    return image


def save_and_show(image, filename):
    path = f'/kaggle/working/slides_output/{filename}.png'
    image.save(path)
    plt.figure(figsize=(16, 9))
    plt.imshow(image)
    plt.axis('off')
    plt.title(filename, fontsize=13, color='white')
    plt.tight_layout()
    plt.show()
    print(f'✅ Saved: {path}')
    return path


print('✅ Helper functions ready!')

✅ Helper functions ready!


## 4. 🖊️ Custom Prompt — Type Your Own!

In [4]:
# ── Interactive Prompt UI ──────────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Controls ───────────────────────────────────────────────────────────────────
prompt_box = widgets.Textarea(
    placeholder='Describe the image... e.g. futuristic city skyline at dusk, glowing blue lights, cinematic, no text',
    layout=widgets.Layout(width='100%', height='100px')
)
neg_prompt_box = widgets.Text(
    placeholder='e.g. blurry, low quality, text, watermark',
    layout=widgets.Layout(width='100%')
)
width_dd   = widgets.Dropdown(options=[512, 768, 1024], value=512,  layout=widgets.Layout(width='100%'))
height_dd  = widgets.Dropdown(options=[512, 768, 1024], value=512,  layout=widgets.Layout(width='100%'))
seed_input = widgets.IntText(value=42,                              layout=widgets.Layout(width='100%'))
steps_sl   = widgets.IntSlider(value=8,  min=1, max=20, step=1,    layout=widgets.Layout(width='100%'))
guidance_sl= widgets.FloatSlider(value=1.0, min=0, max=10, step=0.5, layout=widgets.Layout(width='100%'))
use_pe_cb  = widgets.Checkbox(value=True, description='Use prompt enhancer (use_pe)')
filename_box = widgets.Text(value='my_slide', layout=widgets.Layout(width='100%'))
gen_btn    = widgets.Button(description='🎨 Generate Image', button_style='primary',
                            layout=widgets.Layout(width='100%', height='45px'))
out        = widgets.Output()

# ── Layout ─────────────────────────────────────────────────────────────────────
def row(label, widget):
    return widgets.VBox([widgets.Label(label), widget])

ui = widgets.VBox([
    widgets.HTML('<h3 style="margin:0 0 12px;">🖼 ERNIE-Image-Turbo Generator</h3>'),
    row('Prompt', prompt_box),
    row('Negative prompt (optional)', neg_prompt_box),
    widgets.HBox([
        row('Width',    width_dd),
        row('Height',   height_dd),
        row('Seed',     seed_input),
    ], layout=widgets.Layout(gap='12px')),
    widgets.HBox([
        row('Steps',    steps_sl),
        row('Guidance', guidance_sl),
    ], layout=widgets.Layout(gap='12px')),
    use_pe_cb,
    row('Output filename', filename_box),
    gen_btn,
    out,
], layout=widgets.Layout(gap='10px', padding='16px',
                         border='1px solid #ddd', border_radius='12px'))

# ── Generate ───────────────────────────────────────────────────────────────────
def on_generate(b):
    with out:
        clear_output()
        prompt = prompt_box.value.strip()
        neg    = neg_prompt_box.value.strip()
        if not prompt:
            print('⚠️ Please enter a prompt!')
            return
        print(f'Generating: {filename_box.value}...')
        img = generate_image(
            prompt=prompt,
            seed=seed_input.value,
            width=width_dd.value,
            height=height_dd.value,
        )
        save_and_show(img, filename_box.value)

gen_btn.on_click(on_generate)
display(ui)

In [ ]:
# ── API Server Cell ────────────────────────────────────────────────────────────
!pip install -q flask flask-cors pyngrok

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import threading, base64, io, queue, uuid, time, os

# ── Kill existing ngrok tunnels ────────────────────────────────────────────────
try:
    ngrok.kill()
    time.sleep(2)
except:
    pass

app = Flask(__name__)
CORS(app)

request_queue = queue.Queue()
job_store = {}

def worker():
    while True:
        task = request_queue.get()
        job_id  = task['job_id']
        results = []
        try:
            for i, prompt in enumerate(task['prompts']):
                job_store[job_id]['status'] = f'generating {i+1}/{len(task["prompts"])}'
                img = generate_image(
                    prompt,
                    seed=task['seed'] + i,
                    width=task['width'],
                    height=task['height'],
                )
                buf = io.BytesIO()
                img.save(buf, format='PNG')
                buf.seek(0)
                results.append({
                    'index': i,
                    'image': base64.b64encode(buf.read()).decode('utf-8'),
                    'status': 'ok'
                })
                job_store[job_id]['results'] = results[:]
            job_store[job_id]['status'] = 'done'
        except Exception as e:
            job_store[job_id]['status'] = 'error'
            job_store[job_id]['error'] = str(e)
        finally:
            request_queue.task_done()

threading.Thread(target=worker, daemon=True).start()

@app.route('/generate-batch', methods=['POST'])
def generate_batch():
    data    = request.json
    prompts = data.get('prompts', [])[:5]
    if not prompts:
        return jsonify({'error': 'prompts required'}), 400

    job_id = str(uuid.uuid4())
    job_store[job_id] = {'status': 'queued', 'results': []}
    request_queue.put({
        'job_id':  job_id,
        'prompts': prompts,
        'seed':    data.get('seed', 42),
        'width':   data.get('width', 512),
        'height':  data.get('height', 512),
    })
    return jsonify({'job_id': job_id, 'status': 'queued'})

@app.route('/job/<job_id>', methods=['GET'])
def get_job(job_id):
    job = job_store.get(job_id)
    if not job:
        return jsonify({'error': 'job not found'}), 404
    return jsonify(job)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'running', 'queue': request_queue.qsize()})

# ── ngrok auth token from env var (never hardcode tokens) ─────────────────────
ngrok_token = os.environ.get('NGROK_TOKEN', '')
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)

public_url = ngrok.connect(5000)
print(f'✅ Batch URL: {public_url}/generate-batch')
print(f'   Poll URL:  {public_url}/job/<job_id>')
print(f'   Health:    {public_url}/health')

threading.Thread(target=lambda: app.run(port=5000, use_reloader=False)).start()